# Browser Automation Lecture Notes
Date: 2025-07-09

We'll use this notebook to work through some examples and showcase some essential functions in Playwright.

In [ ]:
# for windows ppl

pip install playwright

Note: you may need to restart the kernel to use updated packages.


In [5]:
!pip install ipykernel==6.28.0

In [7]:
import os
import random
import time

from playwright.async_api import async_playwright, expect

In [8]:
!pip install playwright

  Using cached playwright-1.53.0-py3-none-macosx_11_0_arm64.whl.metadata (3.5 kB)
  Using cached pyee-13.0.0-py3-none-any.whl.metadata (2.9 kB)
Using cached playwright-1.53.0-py3-none-macosx_11_0_arm64.whl (38.6 MB)
Using cached pyee-13.0.0-py3-none-any.whl (15 kB)
  Attempting uninstall: greenlet
    Found existing installation: greenlet 3.0.1
    Uninstalling greenlet-3.0.1:
      Successfully uninstalled greenlet-3.0.1


In [9]:
!which playwright

/opt/anaconda3/bin/playwright


In [10]:
!playwright install

126.9 MiB [                    ] 0% 0.0s126.9 MiB [                    ] 0% 16.0s126.9 MiB [                    ] 0% 18.1s126.9 MiB [                    ] 0% 16.2s126.9 MiB [                    ] 0% 14.1s126.9 MiB [                    ] 0% 14.6s126.9 MiB [                    ] 0% 14.7s126.9 MiB [                    ] 0% 14.5s126.9 MiB [                    ] 1% 13.8s126.9 MiB [                    ] 1% 13.7s126.9 MiB [                    ] 1% 13.3s126.9 MiB [                    ] 1% 13.5s126.9 MiB [                    ] 1% 14.0s126.9 MiB [                    ] 1% 13.6s126.9 MiB [                    ] 1% 13.2s126.9 MiB [                    ] 1% 13.5s126.9 MiB [                    ] 2% 13.3s126.9 MiB [                    ] 2% 12.9s126.9 MiB [                    ] 2% 12.7s126.9 MiB [=                   ] 2% 12.5s126.9 MiB [=                   ] 2% 12.4s126.9 MiB [=                   ] 3% 12.7s126.9 MiB [=                   ] 3% 12.5s126.9 MiB [=                   ] 3% 12.3s126.9 MiB [=     

In [11]:
os.makedirs('data/', exist_ok=True)

In [14]:
# Start the browser
playwright = await async_playwright().start() # await stands for right order of processes 

In [15]:
browser = await playwright.chromium.launch(headless=False)
page = await browser.new_page()
# await browser.close()

In [16]:
async def open_browser(headless=False):
    """
    Starts the automated browser and opens a new window
    """
    # Start playwright
    playwright = await async_playwright().start()

    # Open chromium (chrome) browser, can use firefox or others
    browser = await playwright.chromium.launch(headless=headless)
  
    # Create a new browser window
    page = await browser.new_page()

    return browser, page

In [17]:
driver, page = await open_browser()

In [18]:
# visit a URL
url = 'https://amazon.com'
await page.goto(url)

<Response url='https://www.amazon.com/' request=<Request url='https://www.amazon.com/' method='GET'>>

## XPATH
You learned about beautifulSoup to work with HTML...

Xpath is another way to navigate hierarchical structures of HTML and SVG

It's fast, versatile, and can be used in the developer console and in any computing language.

## Using XPATH
You can test xpaths in the devloper tools under the `console` tab using the function `$x()`. Read about that function [here](https://developer.chrome.com/docs/devtools/console/utilities/#xpath-function).


You can use Playwright's `locator` [function](https://playwright.dev/python/docs/locators#locate-by-css-or-xpath) to find the search box on Amazon site using an xpath or css selector. It'll return the first match unless you add `.all()` to the located elements.

Playwright allows other [locators](https://playwright.dev/python/docs/locators#quick-guide), which the developers suggest (but I don't).

Example of an xpath!

//input[@aria-label="Search Amazon"]

### Finding elements and performing actons
Browser automation is largely about locating elements on the page, and interacting with them in some way.
This can involve filling forms, mocking pressing buttons on a keyboard, clicking things.

In [19]:
# here's how you can do that with xpath
search_bar = page.locator(
    '//input[@aria-label="Search Amazon"]'
)


You can also use the built-in functionality by exploiting the placeholder text
```
page.get_by_placeholder("Search Amazon")
```

In [20]:
search_bar

<Locator frame=<Frame name= url='https://www.amazon.com/'> selector='//input[@aria-label="Search Amazon"]'>

Let's fill in the search bar.

In [21]:
await search_bar.fill('TEST')

In [22]:
search_term = 'womens shirt'
await search_bar.fill(search_term)

You can make the search either by inputting the "Enter" button, or finding the search button and clicking it.

Here's how you can press a [keyboard](https://playwright.dev/docs/api/class-keyboard) button.

In [23]:
await page.keyboard.press("Enter")

Alternatively, you can locate the button perform an [action](https://playwright.dev/python/docs/input#mouse-click) such as a mouse `click`.

In [ ]:
search_button = page.locator(
    '//input[@id="nav-search-submit-button"]'
)
await search_button.click()

In [ ]:
# Katya's solution

product_tiles = await page.locator('//div[@data-component-type="s-search-result"]').all()
len(product_tiles)

60

### Parse the products

For each product, let's print the brand name:

In [34]:
# contains supports substring matches
xpath_product = '//div[contains(@cel_widget_id, "MAIN-SEARCH_RESULTS-")]'
product_tiles = await page.locator(xpath_product).all()
len(product_tiles)

60

Notice we're adding `.all()` to the command, this will return a list, rather than the first element.

If you run all the cells at once, the above will return zero results. This is because the browser needs to await for the element to be visible.

You can force the browser to sleep or check the element is visible using the `expect` [function](https://playwright.dev/python/docs/test-assertions).

In [36]:
# Let's wait for the first product to load
await expect(page.locator(xpath_product).first).to_be_visible()

In [37]:
# run it again, after waiting for the elements to be rendered
product_tiles = await page.locator(xpath_product).all()
len(product_tiles)

60

This is the sleeping method

In [22]:
# import asyncio

# await asyncio.sleep(2)
# product_tiles = await page.locator(xpath_product).all()

Let's parse one product

In [38]:
prod = product_tiles[0]

In [39]:
# see the text of the element
await prod.text_content()

'\n\n\n\n    \n\n\n\n    +12 other colors/patternsFeatured from Amazon brandsFeatured from Amazon brands Amazon EssentialsWomen\'s Regular-Fit 3/4 Sleeve V-Neck T-Shirt (Available in Plus Size), Multipacks 4.2 out of 5 stars 10,520  300+ bought in past monthPrice, product page$8.01$8.01 Typical price: $13.30Typical price: $13.30$13.30Exclusive Prime priceFREE delivery Mon, Jul 14 on $35 of items shipped by AmazonOr fastest delivery Tomorrow, Jul 10  1 sustainability feature<img alt="" src="https://m.media-amazon.com/images/I/11++B3A2NEL.png" height="24px" width="24px"/>  Sustainability featuresThis product has sustainability features recognized by trusted certifications. Safer chemicalsMade with chemicals safer for human health and the environment.As certified by<img alt="" src="https://m.media-amazon.com/images/I/51YvKwF01yL._SS200_.jpg" height="24px" width="24px"/>  OEKO-TEX STANDARD 100Learn more about OEKO-TEX STANDARD 100<img alt="" src="https://m.media-amazon.com/images/I/51YvKwF

Let's iterate through each product and print the brand name, which is saved as a header (h2) with a unique class in the element:

In [48]:
data = []

xpath_brand = '//h2[@class="a-size-mini s-line-clamp-1"]'
xpath_price = '//span[class="a-price-whole"]'

for product in product_tiles:
    brand = await product.locator(xpath_brand).text_content()
    price = await product.locator(xpath_price).text_content()
    row = {
        'brand': brand,
        'price': price
    }
    data.append(row)

data

TimeoutError: Locator.text_content: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("//div[contains(@cel_widget_id, \"MAIN-SEARCH_RESULTS-\")]").first.locator("//span[class=\"a-price-whole\"]")


Although we did this all using Playwright, it's better to save the page source and then parse the saved results in BeautifulSoup, lxml, or whatever parsing software you prefer.

### Annotate the elements we find
Let's find all the ads, and highlight them red on the page.

In [53]:
# xpath_ads = '//div[@data-asin and .//a[@aria-label="View Sponsored information or leave ad feedback"]]'
xpath_ads = '//div[@data-asin and .//span[@aria-label="View Sponsored information or leave ad feedback"]]'
ads = await page.locator(xpath_ads).all()

In [54]:
len(ads)

10

You can "inject" attributes into elements, including style attributes.

In [56]:
elem = ads[0]

In [57]:
style = f"background-color: red !important; transition: all 0.5s linear;"

In [58]:
await elem.evaluate(f"el => el.setAttribute('style','{style}')")

In [59]:
async def stain(elem, color = 'red'):
    """
    Injects a style attribute to stain `elem` the `color` red.
    """
    style = f"background-color: {color} !important; "\
             "transition: all 0.5s linear;"
    await elem.evaluate(f"el => el.setAttribute('style','{style}')")

In [60]:
for elem in ads:
    await stain(elem)

### Get height of document

In [61]:
import pandas as pd

In [62]:
height = await page.evaluate("document.body.scrollHeight")

In [63]:
height

14728

Get the coorindates and size of each element using the `bounding_box` function.

In [64]:
await elem.bounding_box()

{'x': 266.40625, 'y': 1136.96875, 'width': 250.3984375, 'height': 659.4921875}

In [65]:
ad_metadata = []
for elem in ads:
    if await elem.is_visible(): # use this function to only analyze visable elements
        rect = await elem.bounding_box()
        ad_metadata.append(rect)

In [66]:
df = pd.DataFrame(ad_metadata)

In [68]:
df['y']

0   -2695.492188
1   -1564.000000
2   -1564.000000
3    -883.507812
4    -883.507812
5    -203.015625
6    -203.015625
7     456.476562
8     456.476562
9    1136.968750
Name: y, dtype: float64

In [69]:
df['how_far_down'] = df['y'] / height

In [70]:
df.how_far_down.value_counts()

how_far_down
-0.106192    2
-0.059988    2
-0.013784    2
 0.030994    2
-0.183018    1
 0.077198    1
Name: count, dtype: int64

### Save receipts

In [71]:
# how to save what the emulator sees
source = await page.content()
with open('data/amazon_selenium_test.html', 'w') as f:
    f.write(source)

In [72]:
# just what's visible
screenshot = await page.screenshot(path='data/amazon_selenium_test.png')

### Parsing the results however you like
For me it means using lxml, but you can do this same thing in BeautifulSoup, and I encourage you do so...

In [73]:
from lxml import etree

In [74]:
dom = etree.HTML(open('data/amazon_selenium_test.html').read())

In [75]:
product_metadata = []
for result in dom.xpath('.//div[contains(@cel_widget_id, "MAIN-SEARCH_RESULTS")]'):
    # this is where you can parse as many fields as you like.
    brand, product_name = result.xpath('.//h2//text()')[:2]
    product_metadata.append({
        'brand': brand,
        'product_name': product_name
    })

In [76]:
pd.DataFrame(product_metadata)

,brand,product_name
0,Amazon Essentials,Women's Regular-Fit 3/4 Sleeve V-Neck T-Shirt ...
1,AUTOMET,Women Shirts Summer Sweaters Regular Fit Short...
2,Trendy Queen,Womens Basic T Shirts Summer Tops 2025 Crop Sh...
3,AUTOMET,Women's Short Sleeve Shirts Dressy Lace Summer...
4,ANRABESS,Womens Short Sleeve Henley Tops V Neck Dressy ...
5,Blooming Jelly,Women's Dressy Casual Tops Business Work Blous...
6,WIHOLL,Tops for Women Summer Casual Ruffle Trim Sleev...
7,Amazon Essentials,Women's Regular-Fit Short-Sleeve V-Neck T-Shir...
8,AUTOMET,Summer Tops Womens Spring Short Sleeve Shirts ...
9,ANRABESS,Women Short Sleeve V Neck Ribbed Knit Fitted S...


### Automate rotating "AUTOMET"
1. Find all products
2. filter to those with brand == AUTOMET
3. Find the image
    - Inject Javascript to make it spin.

In [80]:
async def spin(elem):
    """
    Injects a style attribute to rotate `elem` 180 degrees.
    """
    style = f"transform: rotate(180deg) !important; "
    await elem.evaluate(
            f"elm => elm.setAttribute('style','{style}')"
    )

Here's how to do this for ads (which we found previously)

In [81]:
for elem in ads:
    await spin(elem)

## Here's the bountry
Ultimately, I want every product image from specific brands (of y0ur choosing) to rotate continuously.
You need a full screenshot or a video is fine, too.

Also, make sure to save the results before you parse them.

In [50]:
for product in product_tiles:
    # get the brand name...
    brand_name = await product.locator(TK)
    
    # check if brand is in list
    if brand_name in brands_to_spin:
        print(product.text_content())
        # find the image TK
        spin(product)

NameError: name 'TK' is not defined

In [ ]:
await driver.close()